## Load the Last.fm Dataset

In [3]:
from implicit.datasets.lastfm import get_lastfm

artists, users, artist_user_plays = get_lastfm()

## Peek Inside the Last.fm Data

In [4]:
print(f"Artists: {len(artists):,}")
print(f"User: {len(users):,}")
print(f"Matrix shape (artists x users): {artist_user_plays.shape}")
print(f"Non-zero interactions: {artist_user_plays.nnz:,}")

Artists: 292,385
User: 358,868
Matrix shape (artists x users): (292385, 358868)
Non-zero interactions: 17,535,606


## Train-Test Split

In [5]:
from implicit.evaluation import train_test_split

# Transpose to (users x artists) format | rows -> users | columns -> items
user_plays = artist_user_plays.T.tocsr()

train_raw, test_raw = train_test_split(user_plays,
                                       train_percentage=0.8,
                                       random_state=42)

print(f"Train shape: {train_raw.shape}, non-zero: {train_raw.nnz:,}")
print(f"Test shape: {test_raw.shape}, non-zero: {test_raw.nnz:,}")

Train shape: (358868, 292385), non-zero: 14,028,047
Test shape: (358868, 292385), non-zero: 3,507,558


## Preprocess Data for Training

- **`bm25_weight`**: Applies BM25 scoring to user-item interactions, reducing the influence of popular items.

- **`csr_matrix`**: A sparse matrix format (Compressed Sparse Row) that stores only non-zero values efficiently.

```python
# Dense matrix (wastes space)
dense = [
    [0, 0, 5],
    [3, 0, 0],
    [0, 2, 0]
]

# CSR format (stores only non-zero)
from scipy.sparse import csr_matrix
sparse = csr_matrix(dense)

print(sparse.data)    # [5, 3, 2] - values
print(sparse.indices) # [2, 0, 1] - columns  
print(sparse.indptr)  # [0, 1, 2, 3] - row starts

## BM25 Weighting for ALS

In [6]:
from implicit.nearest_neighbours import bm25_weight

# ALS: apply BM25 weighting on train_raw
weighted_train_als = bm25_weight(train_raw,     # Sparse matrix (artists x users) with play counts
                                 K1=100,     # Saturation: higher = more weight for repeated plays
                                 B=0.8).tocsr()      # Normalization: higher = penalizes very active users more)

test_als = test_raw.sign()

## Train ALS Model

**Confidence Weight**  
`Confidence = 1 + α × (interaction strength)`

Higher confidence = more important to the model.  
Common `α` values: 1.0 to 40.0.

| Platform | Action | Weight |
|----------|--------|--------|
| E-commerce | Purchase | 10.0 |
| E-commerce | Add to cart | 5.0 |
| E-commerce | Click | 1.0 |
| E-commerce | View | 0.5 |
| YouTube | Watch time (minutes) | 1.0 - 5.0 (scaled) |
| YouTube | Like | 10.0 |
| YouTube | Subscribe | 20.0 |

In [1]:
# from implicit.als import AlternatingLeastSquares

# # Initialize ALS model
# als_model = AlternatingLeastSquares(factors=128,                  # Latent factor dimension (embedding size)
#                                     regularization=0.01,          # Prevents overfitting
#                                     alpha=20.0,                   # confidence weight for positive interactions (plays/listens)
#                                     iterations=50)                # Training epochs


# # Train the model
# als_model.fit(user_items=weighted_train_als,
#               show_progress=True)

## Binary Conversion for BPR and LMF

In [7]:
# converts non-zero to 1
train_binary = train_raw.sign()
test_binary = test_raw.sign()

## Train BPR Model

In [ ]:
# from implicit.bpr import BayesianPersonalizedRanking

# bpr_model = BayesianPersonalizedRanking(factors=100,
#                                         learning_rate=0.01,
#                                         regularization=0.01,
#                                         iterations=100,
#                                         verify_negative_samples=True)

# bpr_model.fit(user_items=train_binary,
#               show_progress=True)

100%|██████████| 100/100 [03:25<00:00,  2.05s/it, train_auc=95.27%, skipped=1.75%]


## Train LMF Model

In [ ]:
# from implicit.lmf import LogisticMatrixFactorization

# lmf_model = LogisticMatrixFactorization(factors=30,
#                                         learning_rate=1.0,
#                                         regularization=0.01,
#                                         iterations=30)

# lmf_model.fit(user_items=train_binary,
#               show_progress=True)

100%|██████████| 30/30 [00:38<00:00,  1.28s/it]


## Models Evaluation

### 🎯 Binary Evaluation (Hits only)

| | |
|---|---|
| **Question** | "Did the user interact with this item at all?" |
| **Best for** | Discovery, new user onboarding, CTR optimization |
| **Fair to** | ✅ BPR / LMF (native format) <br> ⚠️ ALS (slightly handicapped) |

---

### ⚡ Weighted Evaluation (Engagement strength)

| | |
|---|---|
| **Question** | "Did the user strongly engage with this item?" |
| **Best for** | Retention, watch time, play-count-aware recommendations |
| **Fair to** | ✅ ALS (native format) <br> ❌ BPR / LMF (heavily handicapped) |

---

### 📈 Quick Summary

| Strategy | Best For | Fair To |
|----------|----------|---------|
| **Binary** | Discovery, CTR | BPR / LMF |
| **Weighted** | Retention, Engagement | ALS |

---

### 🎯 Final Takeaway

> **For predicting *whether* a user will listen (discovery) → BPR works best.**  
> **For predicting *how much* they'll listen (engagement) → ALS is the right choice.**

**Different questions. Different models. Both valid.**

### Evaluating ALS Model


In [ ]:
# from implicit.evaluation import ranking_metrics_at_k

# als_metrics = ranking_metrics_at_k(als_model, weighted_train_als, test_als, K=10)

100%|██████████| 358511/358511 [03:29<00:00, 1713.42it/s]


### Evaluating BPR Model

In [ ]:
# bpr_metrics = ranking_metrics_at_k(bpr_model, train_binary, test_binary, K=10)

100%|██████████| 358511/358511 [03:23<00:00, 1759.05it/s]


### Evaluating LMF Model

In [ ]:
# lmf_metrics = ranking_metrics_at_k(lmf_model, train_binary, test_binary, K=10)

100%|██████████| 358511/358511 [03:20<00:00, 1788.96it/s]


## Model Performance Comparison

### Metrics Explained

| Metric | Question it answers | Example |
|--------|---------------------|---------|
| **Precision@10** | "Out of 10 recs, how many did user actually like?" | 0.054 = 5.4% → ~5 good recs out of 100 (sparse dataset) |
| **MAP** | "Are the good recs at the TOP of the list?" | 0.02 = low but expected for Last.fm |
| **NDCG** | "Does ranking order make sense? (valuing top positions more)" | 0.05 = model learns something, not random |
| **AUC** | "Given one liked + one disliked, does model pick the liked one?" | 0.50 = coin flip. 0.52 = barely better (normal for sparse data) |


**Why ALS wins for Last.fm:**

| Aspect | ALS | BPR | LMF |
|--------|:---:|:---:|:---:|
| Precision | 5.4% | 6.4% | 3.3% |
| **Play counts matter?** | ✅ Yes | ❌ No | ❌ No |
| **Fair eval on weighted** | ✅ | ❌ | ❌ |

**Final verdict:**
> BPR has higher precision, but ALS answers the **right question** for Last.fm:  
> *"How much will they listen?"* not *"Will they listen at all?"*

In [ ]:
# import pandas as pd

# df = pd.DataFrame([als_metrics, bpr_metrics, lmf_metrics], index=['ALS', 'BPR', 'LMF'])

# df.round(4)

,precision,map,ndcg,auc
ALS,0.0543,0.0206,0.0533,0.5242
BPR,0.0646,0.0281,0.0683,0.5289
LMF,0.0378,0.0143,0.0376,0.5168


## Training Final ALS Model on Full Data
Now that we've validated the model works, we train on **100% of the data** for maximum performance.

In [7]:
from implicit.nearest_neighbours import bm25_weight
from implicit.als import AlternatingLeastSquares

user_plays = artist_user_plays.T.tocsr()

user_plays_weighted = bm25_weight(user_plays, K1=100, B=0.8).tocsr()

# Initialize ALS model
final_als_model = AlternatingLeastSquares(factors=128,           # Latent factor dimension (embedding size)
                                          regularization=0.01,   # Prevents overfitting
                                          alpha=20.0,            # confidence weight for positive interactions (plays/listens)
                                          iterations=50)         # Training epochs


# Train the model
final_als_model.fit(user_items=user_plays_weighted,
                    show_progress=True,)

c:\Users\pouya\AppData\Local\Programs\Python\Python39\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [02:39<00:00,  3.20s/it]


## Examine Learned Factors


**What the numbers mean:**

- Each user has **128 numbers** representing their "taste vector"
- Each artist has **128 numbers** representing their "vibe vector"
- **Similar taste vectors × similar vibe vectors = high recommendation score**

In [8]:
print(f"User factors shape: {final_als_model.user_factors.shape}")  # (n_users, factors)
print(f"Item factors shape: {final_als_model.item_factors.shape}")  # (n_items, factors)

User factors shape: (358868, 128)
Item factors shape: (292385, 128)


### Where `user_factors` comes from

**Only matrix factorization models** (like ALS) have `user_factors`.

#### What it is:
- After training, ALS learns two matrices:
  - `user_factors` - taste vector for each user
  - `item_factors` - vibe vector for each artist

## Example:
```python
model.user_factors.shape  # (n_users, n_factors)
model.user_factors[326348]  # User 326348's taste vector

In [9]:
similar_ids, scores = final_als_model.similar_items(107225, N=50)
for aid, score in zip(similar_ids, scores):
    print(f"{artists[aid]}: {score:.4f}")

eminem: 1.0000
blink-182: 0.9994
kanye west: 0.9994
incubus: 0.9994
britney spears: 0.9993
guns n roses: 0.9993
led zeppelin: 0.9992
the kooks: 0.9991
koЯn: 0.9991
the smashing pumpkins: 0.9989
audioslave: 0.9989
green day: 0.9989
no doubt: 0.9988
system of a down: 0.9988
tenacious d: 0.9988
travis: 0.9988
death cab for cutie: 0.9988
tool: 0.9988
my chemical romance: 0.9987
lily allen: 0.9987
pearl jam: 0.9987
blur: 0.9987
gnarls barkley: 0.9986
the white stripes: 0.9986
john mayer: 0.9986
fall out boy: 0.9986
kaiser chiefs: 0.9986
jimmy eat world: 0.9986
hans zimmer: 0.9986
simon & garfunkel: 0.9986
alice in chains: 0.9985
the who: 0.9984
lady gaga: 0.9984
justin timberlake: 0.9984
the offspring: 0.9984
infected mushroom: 0.9984
bloc party: 0.9983
evanescence: 0.9983
faithless: 0.9983
snow patrol: 0.9983
counting crows: 0.9983
black eyed peas: 0.9983
aerosmith: 0.9982
nelly furtado: 0.9982
sublime: 0.9982
guns n' roses: 0.9982
metallica: 0.9982
beck: 0.9982
black sabbath: 0.9981
franz

In [10]:
# 100th user embedding vector 
final_als_model.user_factors[99]

array([ -3.6194928 ,   2.4342268 ,   7.6704264 , -11.010006  ,
         7.2297664 , -16.317196  ,  -0.3145261 , -12.204365  ,
        11.886114  ,   3.6512878 ,  -6.8279    ,   6.268649  ,
         5.41042   ,   8.157794  ,  -2.7939458 ,   5.3657265 ,
         4.6882553 ,   3.5092638 ,   1.5898571 ,  -9.237634  ,
         2.942089  ,  -0.49555147,  12.38149   ,  -0.48947087,
        -0.26577145,  -6.3945546 ,  -3.8150957 ,  -1.4090316 ,
        -4.4589477 ,  -4.2345695 ,  -3.1633108 ,   4.003531  ,
         0.33312687,   1.7751539 ,  -2.0089576 ,  -8.087628  ,
        13.589363  ,  15.021113  ,   0.54448736,   6.7619762 ,
        -4.069596  ,   2.0225384 ,  -5.358516  ,   6.5066648 ,
        -4.714345  ,  -3.3673735 ,  13.644183  ,  -5.79359   ,
         0.18407822, -19.638706  ,  -7.582425  ,   0.7470741 ,
        -4.8907366 ,   1.2783076 ,   0.15594342,   5.821674  ,
         2.3064759 ,   0.09909759,  11.611429  ,   1.9377104 ,
         0.07423488,  13.816661  ,   0.54206634,  -4.66

## Make Recommendations

In [78]:
# Select a specific user (by index position)
user_id = 99

# Get user's play history (artists and weighted counts) - model automatically excludes these from recommendations
artist_ids, scores = final_als_model.recommend(userid=user_id,
                                 user_items=user_plays_weighted[user_id],
                                 N=10,
                                 filter_already_liked_items=True)

# Loop through both arrays simultaneously
for artist_id, score in zip(artist_ids, scores):    # zip pairs them together
    print(f"{artists[artist_id]}: {score:.4f}")

ill niño: 1.5111
ferry corsten: 1.4228
atb: 1.4203
ayreon: 1.4019
oceanlab: 1.3908
paul van dyk: 1.3803
era: 1.3453
chris rea: 1.2572
tiësto: 1.2531
markus schulz: 1.2483


## Exploring a User's Liked Artists

```python
# Full sparse row representation:
user_plays[user_id]
#   (artist_ID,    play_count)
#   (157195,       25.0)       ← Eminem
#   (3853,         50.0)       ← 50 Cent  
#   (12345,        10.0)       ← Dr. Dre

# .indices extracts ONLY the artist IDs:
liked_artists = user_plays[user_id].indices
# Result: [157195, 3853, 12345]

# .data would extract ONLY the play counts:
play_counts = user_plays[user_id].data
# Result: [25.0, 50.0, 10.0]

In [12]:
# Select a specific user (by index position)
user_id = 100

# Get all artist IDs that this user has listened to (non-zero interactions)
liked_artists_id = user_plays[user_id].indices

# Loop through the first 10 artist IDs the user listened to
for artist_id in liked_artists_id[:10]:
    
    # Convert artist ID to actual name and print it
    print(artists[artist_id])

almodóvar, pedro
antonio carlos jobim & astrud gilberto
antony and the johnsons
beastie boys
beirut
benjamin biolay
best of
black eyed peas
christina rosenvinge
cocorosie


### Why ALS recommends Rock for Eminem fans

**ALS doesn't understand music genres.** It only finds patterns in listening behavior.

- **Eminem listeners** → also listen to **Incubus, The Killers** (rock bands)
- **Model assumption:** If users listen together, they're "similar"

### The Problem: Popularity Bias

Popular artists cluster together because **everyone** listens to them, regardless of genre. `(popularity bias)`

| You asked for | ALS gave you |
|--------------|--------------|
| Music that sounds like Rap | Music that Rap fans also listen to (Rock/Alternative) |

**ALS prioritizes co-occurrence ("fans also like") over audio features (genre).**

## Artist Similarity Function


In [79]:
def artists_fans_also_like(artist_name: str, N: int = 10):
    """Return artists frequently listened to by the same users."""

    # Convert artist name to its ID in the artists list
    artist_id = list(artists).index(artist_name)
    
    # Find top N similar artists using ALS item factors
    similar_ids, similar_scores = final_als_model.similar_items(itemid=artist_id, N=N)
    
    # Print each similar artist with their similarity score
    for id, score in zip(similar_ids, similar_scores):
        print(f"{artists[id]}: {score:.4f}")

artists_fans_also_like("jay-z")

jay-z: 1.0000
outkast: 0.9979
snoop dogg: 0.9974
2pac: 0.9972
kanye west: 0.9963
justin timberlake: 0.9953
gnarls barkley: 0.9951
eminem: 0.9948
m.i.a.: 0.9946
lily allen: 0.9946


## Find Artist ID Function

In [ ]:
def find_artist_id(artist_name: str):
    """Find and display the ID of an artist by their name."""
    try:
        artist_id = list(artists).index(artist_name.lower())
        print(f"{artists[artist_id]} --> {artist_id}")
        
    except ValueError:
        print(f"❌ Artist '{artist_name}' not found. Try checking spelling.")

find_artist_id("NaS")

nas --> 196221


## User Recommendation Function

In [ ]:
def get_user_recommendations(user_id: int, n_recs: int = 10, n_history: int = 10):
    """Display user's liked artists + personalized recommendations"""
    
    # Get artists IDs user has listend to
    already_liked_artists_id = user_plays[user_id].indices

    # Convert IDs to artist names
    liked_artists = [artists[artid] for artid in already_liked_artists_id[]]
    
    # Generate recommendations (excludes already liked artists)
    artist_rec_id, scores = final_als_model.recommend(userid=user_id,
                                               user_items=user_plays_weighted[user_id],
                                               N=n_recs,
                                               filter_already_liked_items=True)
    
    # Display user's listening history
    print(30 * "-")    
    print(f"User {user_id} already liked artists:")
    print(30 * "-")
    for artist in liked_artists[:n_history]:
        print(artist)
    
    print("\n")
    
    # Display recommendations
    print(30 * "-")
    print(f"User {user_id} recommended artists:")
    print(30 * "-")
    for id in artist_rec_id:
        print(artists[id])

get_user_recommendations(user_id=18650)

------------------------------
User 18650 already liked artists:
------------------------------
50 cent
all that remains
amorphis
billy talent
blind guardian
chris hülsbeck
cky
depeche mode
die fantastischen vier
die Ärzte


------------------------------
User 18650 recommended artists:
------------------------------
scooter
schandmaul
atb
apoptygma berzerk
saltatio mortis
sentenced
and one
kmfdm
corvus corax
crematory


## Last.fm Recommendation API

In [ ]:
def get_user_recommendations(user_id: int, n_recs: int = 10, n_history: int = 10):
    """Display user's liked artists + personalized recommendations"""
    
    # Get artists IDs user has listend to
    already_liked_artists_id = user_plays[user_id].indices

    # Convert IDs to artist names
    liked_artists = [artists[artid] for artid in already_liked_artists_id[:n_history]]
    
    # Generate recommendations (excludes already liked artists)
    artist_rec_id, scores = final_als_model.recommend(userid=user_id,
                                               user_items=user_plays_weighted[user_id],
                                               N=n_recs,
                                               filter_already_liked_items=True)
    
    # List comprehension: map each artist ID to its name
    recommended_artists = [artists[id] for id in artist_rec_id]

    return {
        "user_id": user_id,
        "liked_artists" : liked_artists,
        "recommendations": recommended_artists
    }

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app = FastAPI()

# Define response data structure
class UserRecommendations(BaseModel):
    user_id: int
    liked_artists: List[str]
    recommendations: List[str]

# API endpoint
@app.get("/recommendations/{user_id}", response_model=UserRecommendations)
def recommendations(user_id: int):
    return get_user_recommendations

## Saving and Loading ALS Model

### Saving ALS Models: `implicit` vs `joblib`

When using the `implicit` library, saving and loading models can be a bit inconvenient because the model alone is not fully self-contained. You usually need to manually save and restore additional components such as:

- user/item mappings
- preprocessing steps (e.g., BM25 weighting)
- sparse interaction matrices

This makes reloading the model more complex and error-prone.

### Problem with `implicit` save/load

Even if you save the model like this:

```python
final_als_model.save("lastfm_als.npz")
````

You still need to manually handle:

* Recreating the model with the same hyperparameters
* Reloading mappings (user_id → index, item_id → index)
* Reapplying preprocessing (BM25, normalization, etc.)

So the model is **not fully portable by itself**.

---

## Simpler approach: `joblib`

Using `joblib`, you can serialize the entire model object directly.


```python
import joblib

# Save model
joblib.dump(final_als_model, "lastfm_als_model.pkl")

# Load model
loaded_model = joblib.load("lastfm_als_model.pkl")

```

---

## Making recommendations after loading

```python
loaded_model.recommend(
    userid=18650,
    user_items=user_plays_weighted[18650],
    N=10
)
```

---

## Why `joblib` is easier

* Saves the **entire Python object**
* No need to manually recreate model structure
* No need to separately load mappings or configs
* Faster and more convenient for ML experimentation

---

## Trade-off

| Method   | Pros                           | Cons                                |
| -------- | ------------------------------ | ----------------------------------- |
| implicit | Optimized, lightweight format  | Requires manual reconstruction      |
| joblib   | Easy save/load, fully portable | Larger files, less explicit control |

---

## Summary

* `implicit.save()` → efficient but requires rebuilding pipeline manually
* `joblib.dump()` → simpler and more practical for development and experiments

```
```


In [99]:
import joblib

# Saving
joblib.dump(final_als_model, "lastfm_als_model.pkl")

# Loading
loaded_model = joblib.load("lastfm_als_model.pkl")
loaded_model.recommend(userid=18650, user_items=user_plays_weighted[18650], N=10)

(array([230476, 230118,  33980,  29469, 228145, 231990,  23382, 162316,
         76294,  77504]),
 array([1.4292376, 1.3538417, 1.3151144, 1.3054953, 1.2922467, 1.2570194,
        1.242332 , 1.2331338, 1.2234799, 1.2169433], dtype=float32))